### In case of IED, there is o need to use these lines of code

* Dataset link: https://datacommons.cyverse.org/browse/iplant/home/shared/commons_repo/curated/GenomesToFields_2014_2017_v1/G2F_Planting_Season_2017_v1

In [46]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
import os
os.listdir('/content/drive/My Drive')

['DriveSharer',
 'Kolop',
 'GDToT',
 'me.jpg',
 'WhatsApp Image 2024-12-20 at 22.40.37_fcdc2f8a.jpg',
 'Colab Notebooks',
 'WhatsApp Image 2025-02-08 at 20.49.09_02e7cb53.jpg',
 'Copy of VC_RedistInstaller.exe',
 'To-do list.gsheet',
 'Google sheets',
 'Untitled spreadsheet.gsheet',
 'changed data type(numeric) (2).xlsx',
 'changed data type(numeric) (1).xlsx',
 'changed data type(numeric).xlsx',
 'changed data type(numeric).gsheet',
 'df_78_data.xlsx',
 'pratice sheet.gsheet',
 'Ask Gemini about the role of pyhton in plant bree... (5).gdoc',
 'Ask Gemini about the role of pyhton in plant bree... (4).gdoc',
 'Ask Gemini about the role of pyhton in plant bree... (3).gdoc',
 'Ask Gemini about the role of pyhton in plant bree... (2).gdoc',
 'Ask Gemini about the role of pyhton in plant bree... (1).gdoc',
 'Ask Gemini about the role of pyhton in plant bree....gdoc',
 'Emails to different stakeholders.gdoc',
 'mmem.jpg',
 'Tableau',
 'Learner-facing C3 Automatidata dataset for Tableau proje

In [49]:
dataset_dir = ('/content/drive/My Drive/Dl Models/geno_pheno')
os.listdir(dataset_dir)

['maize-yield-prediction.ipynb',
 'g2f_2017_hybrid_data_clean.csv',
 'g2f_2017_ZeaGBSv27_Imputed_AGPv4.h5',
 'g2f_2017_gbs_hybrid_codes.xlsx',
 'g2f_2017_weather_data.csv',
 'g2f_2016_hybrid_data_clean.csv',
 'g2f_2016_weather_data.csv',
 'g2f_2015_hybrid_data_clean.csv',
 'g2f_2015_weather.csv',
 'g2f_2014_hybrid_data_clean.csv',
 'g2f_2014_weather.csv',
 'results.png',
 'best_model.pk1',
 'X_snp.npy',
 'pca.pkl',
 'scaler_final.pkl',
 'y.npy',
 'X_env.npy',
 'X_final.npy',
 'scaler_snp.pkl',
 'df_raw.csv',
 'best_model.pkl']

In [50]:
# importing required libraries for pre-preprocessing of project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import pickle
import os
import warnings

In [51]:
# ── Check what files exist in Drive ───────────────────
import os

BASE = '/content/drive/My Drive/Dl Models/geno_pheno/'

print("Files in Drive:")
for f in sorted(os.listdir(BASE)):
    size = os.path.getsize(BASE + f) / 1024 / 1024
    print(f"  {f:35} {size:.1f} MB")

Files in Drive:
  X_env.npy                           0.2 MB
  X_final.npy                         0.3 MB
  X_snp.npy                           109.4 MB
  best_model.pk1                      4.4 MB
  best_model.pkl                      4.4 MB
  df_raw.csv                          2.9 MB
  g2f_2014_hybrid_data_clean.csv      2.0 MB
  g2f_2014_weather.csv                28.8 MB
  g2f_2015_hybrid_data_clean.csv      1.8 MB
  g2f_2015_weather.csv                34.5 MB
  g2f_2016_hybrid_data_clean.csv      2.5 MB
  g2f_2016_weather_data.csv           40.7 MB
  g2f_2017_ZeaGBSv27_Imputed_AGPv4.h5 532.7 MB
  g2f_2017_gbs_hybrid_codes.xlsx      0.1 MB
  g2f_2017_hybrid_data_clean.csv      2.7 MB
  g2f_2017_weather_data.csv           38.2 MB
  maize-yield-prediction.ipynb        1.8 MB
  pca.pkl                             0.4 MB
  results.png                         0.0 MB
  scaler_final.pkl                    0.0 MB
  scaler_snp.pkl                      0.2 MB
  y.npy                        

In [52]:
# importing datasets from google drive
BASE = '/content/drive/My Drive/Dl Models/geno_pheno/'
PHE = BASE +  'g2f_2017_hybrid_data_clean.csv'
GEN = BASE +  'g2f_2017_ZeaGBSv27_Imputed_AGPv4.h5'
WEA = BASE + 'g2f_2017_weather_data.csv'

print('Importing datasets is completed')

Importing datasets is completed


In [53]:
# Loading datasets
df_phe = pd.read_csv(PHE, low_memory=False)
df_wea = pd.read_csv(WEA, low_memory=False)

with h5py.File(GEN, 'r') as f:
  n_taxa = len([k for k in f['Taxa'].keys()
            if k !='TaxaOrder'])
  n_snps = f['Positions']['Positions'].shape[0]

print('Loading datasets:')
print('Phenotype:', df_phe.shape[0])
print('Weather:', df_wea.shape[0])
print('Plant Lines:', n_taxa)
print('SNP Markers:', n_snps)

# Yield distribution
fig = px.histogram(
    df_phe.dropna(subset=['Grain Yield [bu/A]']),
    x='Grain Yield [bu/A]',
    nbins=40,
    title='Raw Yield Distribution (All Plots)',
    color_discrete_sequence=['steelblue'],
    template='plotly_white'
)
fig.update_layout(height=400)
fig.show()

Loading datasets:
Phenotype: 18381
Weather: 253994
Plant Lines: 1577
SNP Markers: 945574


### EDA of datasets

In [54]:
# cleaning taxon ID columns names
def clean_id(x):
  if pd.isna(x):
    return None
  return str(x).replace('(', '').replace(')', '').split(':')[0].strip()

# case insenstive parent name to genotype ID matching

def match_id(x):
  if pd.isna(x):
    return None
  cleaned = clean_id(str(x))
  if cleaned is None:
    return None
  return GENO_LOWER.get(cleaned.lower(), None)

# building genotype lookups
with h5py.File(GEN, 'r') as f:
  taxa_raw = [k for k in f['Taxa'].keys()
              if k !='TaxaOrder']

taxa_clean = [clean_id(t) for t in taxa_raw]
GENO_LOWER = {g.lower(): g for g in taxa_clean if g}

TAXA_LOOKUP = {}
with h5py.File(GEN, 'r') as f:
  for key in f['Genotypes'].keys():
    c = clean_id(key)
    if c and c not in TAXA_LOOKUP:
      TAXA_LOOKUP[c] = key

print(f"PLant lines indexed: {len(GENO_LOWER):,}")
print(f"Genotype entries: {len(TAXA_LOOKUP):,}")



PLant lines indexed: 1,321
Genotype entries: 1,327


In [55]:
# Preparing phenotypes datasets
COLS = [
    'Pedigree', 'Field-Location', 'Grain Yield [bu/A]',
    'Plant Height [cm]', 'Grain Moisture [%]', 'Silk DAP [days]',
    'Pollen DAP [days]', 'Ear Height [cm]'
]

df = df_phe[COLS].dropna(subset=['Grain Yield [bu/A]']).copy()

# spliting pedigree into parents
df[['Parent1', 'Parent2']] = (df['Pedigree'].str.split('/', expand=True))

# matching parents to genotype
df['clean_p1'] = df['Parent1'].apply(match_id)
df['clean_p2'] = df['Parent2'].apply(match_id)

# keeping rows where both parents have genotype data
df = df[
    (df['Plant Height [cm]'] > 0) &
    (df['Silk DAP [days]']   > 0) &
    (df['Pollen DAP [days]'] > 0) &
    (df['Ear Height [cm]']   > 0)
].reset_index(drop=True)

# encode location
df['location_code'] = pd.factorize(df['Field-Location'])[0]

print('Phenotype prepared:')
print(f'Rows: {df.shape[0]:,}')
print(f"Hybrids: {df['Pedigree'].nunique():,}")
print(f"Locations: {df['Field-Location'].nunique():,}")
print(df[['Pedigree', 'Grain Yield [bu/A]','clean_p1', 'clean_p2']].head())

Phenotype prepared:
Rows: 11,746
Hybrids: 654
Locations: 23
                  Pedigree  Grain Yield [bu/A] clean_p1 clean_p2
0                DKC 67-88               24.19     None     None
1              PHP38/PHN47               20.50    PHP38    PHN47
2             2369/LH123HT               29.84     None  LH123Ht
3  NILASQ4G71I03S2/LH123HT               23.87     None  LH123Ht
4               CGR01/LH82               70.78    CGR01     None


In [56]:
# Plot: Yield by Location
fig = px.box(
    df,
    x='Field-Location',
    y='Grain Yield [bu/A]',
    color='Field-Location',
    title='Yield Distribution by Location',
    template='plotly_white'
)
fig.update_layout(
    showlegend=False,
    xaxis_tickangle=-45,
    height=500
)
fig.show()

# Yield vs Plant Height
fig2 = px.scatter(
    df,
    x='Plant Height [cm]',
    y='Grain Yield [bu/A]',
    color='Field-Location',
    title='Yield vs Plant Height',
    template='plotly_white',
    opacity=0.5
)
fig2.update_layout(height=450)
fig2.show()

In [57]:
# process weather
# renaming location columns
df_wea = df_wea.rename(columns={'Field Location':'Field-Location'})

# weather columns to use
WEA_COLS = [
    'Temperature [C]', 'Relative Humidity [%]',
    'Rainfall [mm]', 'Solar Radiation [W/m2]',
    'Wind Speed [m/s]', 'Photoperiod [hours]'
]
# converitng to numeric
df_wea['Month'] = pd.to_numeric(df_wea['Month'], errors='coerce')
for col in WEA_COLS:
    if col in df_wea.columns:
        df_wea[col] = pd.to_numeric(df_wea[col], errors='coerce')


# growing season
w_season = df_wea[df_wea['Month'].between(5, 9)].copy()
wea_season = w_season.groupby('Field-Location').agg({
    'Temperature [C]':'mean',
    'Relative Humidity [%]':'mean', 'Rainfall [mm]':'sum',
    'Solar Radiation [W/m2]':'mean', 'Wind Speed [m/s]':'mean', 'Photoperiod [hours]':'mean'
}).reset_index()
# critical period
w_crit = df_wea[df_wea['Month'].between(6, 8)].copy()
wea_crit = w_crit.groupby('Field-Location').agg({
    'Temperature [C]':['mean', 'max'],
    'Rainfall [mm]':'sum', 'Solar Radiation [W/m2]':'mean',
    'Relative Humidity [%]':'mean'
}).reset_index()

wea_crit.columns = [
    'Field-Location', 'Temp_mean_crit', 'Temp_max_crit',
    'Rain_crit', 'Solar_crit', 'Humid_crit'
]

print('Weather processed:')
print(f'Season locations: {wea_season.shape[0]}')
print(f"Critical locations: {wea_crit.shape[0]}")

Weather processed:
Season locations: 25
Critical locations: 25


In [58]:
# Temperature by Location
fig = px.bar(
    wea_season.sort_values('Temperature [C]',
                            ascending=False),
    x='Field-Location',
    y='Temperature [C]',
    color='Temperature [C]',
    color_continuous_scale='RdYlGn_r',
    title='Average Growing Season Temperature by Location',
    template='plotly_white'
)
fig.update_layout(
    xaxis_tickangle=-45,
    height=450,
    showlegend=False
)
fig.show()

# Rainfall by Location
fig2 = px.bar(
    wea_season.sort_values('Rainfall [mm]',
                            ascending=False),
    x='Field-Location',
    y='Rainfall [mm]',
    color='Rainfall [mm]',
    color_continuous_scale='Blues',
    title='Total Growing Season Rainfall by Location',
    template='plotly_white'
)
fig2.update_layout(
    xaxis_tickangle=-45,
    height=450,
    showlegend=False
)
fig2.show()

In [59]:
# merging weather
ped_loc = df_phe[['Pedigree', 'Field-Location']].dropna().drop_duplicates('Pedigree').reset_index(drop=True)

# merge
df = df.merge(ped_loc, on='Pedigree', how='left',
              suffixes=('', '_loc'))
if 'Field-Location_loc' in df.columns:
  df = df.drop(columns=['Field-Location_loc'])

df = df.merge(wea_season, on='Field-Location', how='left')
df = df.merge(wea_crit, on='Field-Location', how='left')

# fill missing values in dataset with mean
wea_all_cols = (WEA_COLS+['Temp_mean_crit', 'Temp_max_crit', 'Rain_crit',
                          'Solar_crit', 'Humid_crit'])

for col in wea_all_cols:
  if col in df.columns:
    df[col] = df[col].fillna(df[col].mean())

print('Weather merger')
print(f'Final shape: {df.shape}')
print(f"Nan count: {df[wea_all_cols].isna().sum().sum()}")


Weather merger
Final shape: (11746, 24)
Nan count: 0


In [60]:
# Yield vs Temperature
fig = px.scatter(
    df,
    x='Temperature [C]',
    y='Grain Yield [bu/A]',
    color='Field-Location',
    trendline='ols',
    title='Yield vs Growing Season Temperature',
    template='plotly_white',
    opacity=0.5
)
fig.update_layout(height=450)
fig.show()

# Yield vs Rainfall
fig2 = px.scatter(
    df,
    x='Rainfall [mm]',
    y='Grain Yield [bu/A]',
    color='Field-Location',
    trendline='ols',
    title='Yield vs Growing Season Rainfall',
    template='plotly_white',
    opacity=0.5
)
fig2.update_layout(height=450)
fig2.show()

In [61]:
# packages for model training and saving

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score


In [62]:
# loading SNPs
# feature columns for model
FEAT_COLS = [
    'Plant Height [cm]','Grain Moisture [%]', 'Silk DAP [days]',
    'Pollen DAP [days]', 'Ear Height [cm]', 'location_code',
    'Temperature [C]', 'Relative Humidity [%]', 'Rainfall [mm]', 'Solar Radiation [W/m2]',
    'Wind Speed [m/s]', 'Photoperiod [hours]', 'Temp_mean_crit', 'Temp_max_crit', 'Rain_crit',
    'Solar_crit', 'Humid_crit'
]

N_SNPS = 5000

X_female, X_male, y, X_env = [], [], [], []

with h5py.File(GEN, 'r') as f:
  total = len(df)
  for i, (_, row) in enumerate(df.iterrows()):
    if i % 5000 == 0:
      print(f"{i:,}/{total:,} rows processed")

    p1 = row['clean_p1']
    p2 = row['clean_p2']

    if p1 in TAXA_LOOKUP and p2 in TAXA_LOOKUP:
      snp1 = f['Genotypes'][TAXA_LOOKUP[p1]]['calls'][:N_SNPS].astype(np.float32)
      snp2 = f['Genotypes'][TAXA_LOOKUP[p2]]['calls'][:N_SNPS].astype(np.float32)
      snp1[snp1==-1]=0
      snp2[snp2==-1]=0

      X_female.append(snp1)
      X_male.append(snp2)
      y.append(row['Grain Yield [bu/A]'])
      X_env.append([float(row[c])
                          if pd.notna(row[c]) else 0.0
                          for c in FEAT_COLS])

X_female = np.array(X_female, dtype=np.float32)
X_male = np.array(X_male, dtype=np.float32)
X_env = np.array(X_env, dtype=np.float32)
y = np.array(y, dtype=np.float32)
X_snp = np.concatenate([X_female, X_male], axis=1)

print('SNPs data loaded')
print(f"Samples: {len(y):,}")
print(f"SNP shape: {X_snp.shape}")
print(f"Env shape: {X_env.shape}")

0/11,746 rows processed
5,000/11,746 rows processed
10,000/11,746 rows processed
SNPs data loaded
Samples: 2,867
SNP shape: (2867, 10000)
Env shape: (2867, 17)


In [63]:
# Yield Distribution Final Dataset
fig = px.histogram(
    x=y, nbins=40,
    title='Yield Distribution (Final Dataset)',
    labels={'x': 'Grain Yield (bu/A)'},
    color_discrete_sequence=['steelblue'],
    template='plotly_white'
)
fig.add_vline(
    x=y.mean(), line_dash='dash',
    line_color='red',
    annotation_text=f'Mean: {y.mean():.1f}'
)
fig.update_layout(height=400)
fig.show()


In [64]:
# PCA and final features
N_COMP = 10

# scaling SNPs
scaler_snp = StandardScaler()
X_snp_sc = scaler_snp.fit_transform(X_snp)

# compress with PCA
pca = PCA(n_components=N_COMP, random_state=42)
X_snp_pca = pca.fit_transform(X_snp_sc)

# combing PCA and enviorment
X_final = np.concatenate([X_snp_pca, X_env], axis=1)
print('Features engineered')
print(f"PCA variance explained:"
      f"{pca.explained_variance_ratio_.round(3)}")
print(f"Total variance:"
      f"{pca.explained_variance_ratio_.sum():.3f}")
print(f'SNP components: {N_COMP}')
print(f"Env features: {X_env.shape[1]}")
print(f"Total features: {X_final.shape[1]}")

Features engineered
PCA variance explained:[0.102 0.095 0.056 0.051 0.042 0.041 0.036 0.033 0.03  0.027]
Total variance:0.513
SNP components: 10
Env features: 17
Total features: 27


In [65]:
# PCA Variance Explained
variance = pca.explained_variance_ratio_ * 100
cumvar = np.cumsum(variance)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=[f'PCA_{i+1}' for i in range(N_COMP)],
        y=variance,
        name='Individual Variance',
        marker_color='steelblue'
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=[f'PCA_{i+1}' for i in range(N_COMP)],
        y=cumvar,
        name='Cumulative Variance',
        line=dict(color='red', width=2),
        mode='lines+markers'
    ),
    secondary_y=True
)
fig.update_layout(
    title='PCA Variance Explained by Component',
    template='plotly_white',
    height=450
)
fig.update_yaxes(
    title_text='Individual Variance (%)',
    secondary_y=False
)
fig.update_yaxes(
    title_text='Cumulative Variance (%)',
    secondary_y=True
)
fig.show()


In [66]:
# Training model

# split data
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

# scaling
scaler_final = StandardScaler()
X_train_sc = scaler_final.fit_transform(X_train)
X_test_sc = scaler_final.transform(X_test)

# train random forest
rf = RandomForestRegressor(n_estimators=200, max_depth=10,min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)

# evaluation
y_pred = rf.predict(X_test_sc)
test_r2 = r2_score(y_test, y_pred)

# cross validation
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    ))
])

cv_scores = cross_val_score(
    pipe, X_final, y, cv=5, scoring='r2'
)
print('Model tranined')
print(f'test r2: {test_r2:.3f}')
print(f"CV mean r2: {cv_scores.mean():.3f}")
print(f'CV std: {cv_scores.std():.3f}')


Model tranined
test r2: 0.635
CV mean r2: 0.573
CV std: 0.110


In [67]:

# Actual vs Predicted
df_pred = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred,
    'Error': np.abs(y_test - y_pred).round(2)
})

fig = px.scatter(
    df_pred,
    x='Actual', y='Predicted',
    color='Error',
    color_continuous_scale='RdYlGn_r',
    hover_data=['Error'],
    title=f'Actual vs Predicted Yield (R²={test_r2:.3f})',
    labels={
        'Actual': 'Actual Yield (bu/A)',
        'Predicted': 'Predicted Yield (bu/A)'
    },
    template='plotly_white',
    opacity=0.7
)
fig.add_trace(go.Scatter(
    x=[y_test.min(), y_test.max()],
    y=[y_test.min(), y_test.max()],
    mode='lines',
    name='Perfect',
    line=dict(color='red', dash='dash', width=2)
))
fig.update_layout(height=500)
fig.show()

# Residuals
residuals = y_test - y_pred
fig2 = px.scatter(
    x=y_pred, y=residuals,
    color=np.abs(residuals),
    color_continuous_scale='RdYlGn_r',
    title='Residual Plot',
    labels={
        'x': 'Predicted Yield (bu/A)',
        'y': 'Residual (Actual - Predicted)'
    },
    template='plotly_white',
    opacity=0.6
)
fig2.add_hline(
    y=0, line_dash='dash',
    line_color='red', line_width=2
)
fig2.update_layout(height=450)
fig2.show()

# CV Scores
fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=[f'Fold {i+1}' for i in range(5)],
    y=cv_scores,
    marker_color=['green' if s > 0 else 'red'
                  for s in cv_scores],
    text=[f'{s:.3f}' for s in cv_scores],
    textposition='outside'
))
fig3.add_hline(
    y=cv_scores.mean(),
    line_dash='dash', line_color='blue',
    annotation_text=f'Mean: {cv_scores.mean():.3f}'
)
fig3.update_layout(
    title='Cross Validation R² Scores (5-fold)',
    yaxis_title='R² Score',
    template='plotly_white',
    height=400
)
fig3.show()

In [68]:
# Feature importance analysis

feat_names  = ([f'PCA_{i+1}' for i in range(N_COMP)]
               + FEAT_COLS)
importances = rf.feature_importances_

df_imp = pd.DataFrame({
    'Feature':    feat_names,
    'Importance': importances,
    'Type':       (
        ['Genetics (PCA)'] * N_COMP +
        ['Plant Trait']    * 6      +
        ['Season Weather'] * 6      +
        ['Critical Weather'] * 5
    )
}).sort_values('Importance', ascending=True)

# Feature Importance
fig = px.bar(
    df_imp,
    x='Importance', y='Feature',
    color='Type',
    orientation='h',
    title='Feature Importance — What Drives Yield?',
    template='plotly_white',
    color_discrete_map={
        'Genetics (PCA)': 'steelblue',
        'Plant Trait': 'green',
        'Season Weather': 'orange',
        'Critical Weather': 'red'
    }
)
fig.update_layout(height=700, legend_title='Feature Type')
fig.show()

# Importance by Category
df_cat = df_imp.groupby('Type')['Importance'].sum().reset_index()

fig2 = px.pie(
    df_cat,
    values='Importance',
    names='Type',
    title='Feature Importance by Category',
    color_discrete_map={
        'Genetics (PCA)': 'steelblue',
        'Plant Trait': 'green',
        'Season Weather': 'orange',
        'Critical Weather': 'red'
    },
    template='plotly_white'
)
fig2.update_layout(height=450)
fig2.show()

print("\nImportance by category:")
for _, row in df_cat.sort_values(
    'Importance', ascending=False
).iterrows():
    print(f"{row['Type']:25} {row['Importance']:.3f}")


Importance by category:
Plant Trait               0.431
Season Weather            0.319
Genetics (PCA)            0.184
Critical Weather          0.066


In [69]:
# saving model
np.save(BASE+'X_snp.npy', X_snp)
np.save(BASE+'X_env.npy', X_env)
np.save(BASE+'X_final.npy', X_final)
np.save(BASE+'y.npy', y)
df.to_csv(BASE+'df_raw.csv', index=False)

pickle.dump(rf, open(BASE+'best_model.pkl', 'wb'))
pickle.dump(pca, open(BASE+'pca.pkl', 'wb'))
pickle.dump(scaler_snp, open(BASE+'scaler_snp.pkl', 'wb'))
pickle.dump(scaler_final, open(BASE+'scaler_final.pkl', 'wb'))

print('Model saved to drive')
print(f'Samples: {len(y):,}')
print(f"Features: {X_final.shape[1]}")
print(f"Test R2: {test_r2:.3f}")
print(f"CV mean R2: {cv_scores.mean():.3f}")

Model saved to drive
Samples: 2,867
Features: 27
Test R2: 0.635
CV mean R2: 0.573


In [70]:
# Prediction functions
def predict_yield(parent1, parent2, location, verbose=True):
  p1 = match_id(parent1)
  p2 = match_id(parent2)

  if p1 is None:
    print(f'"{parent1}" not in genotype database')
    return None
  if p2 is None:
    print(f"'{parent2}' not in genotype database")
    return None
  if p1 not in TAXA_LOOKUP:
    print(f"'{parent1}' not in taxa lookup")
    return None
  if p2 not in TAXA_LOOKUP:
    print(f"'{parent2}' not in taxa lookup")
    return None

  loc_data = df[df['Field-Location'] == location]
  if len(loc_data) == 0:
    print(f"Location '{location}' not found")
    print(f"Available:"
          f"{sorted(df['Field-Location'].unique())}")
    return None
  with h5py.File(GEN, 'r') as f:
    snp1 = f['Genotypes'][TAXA_LOOKUP[p1]]['calls'][:N_SNPS].astype(np.float32)
    snp2 = f['Genotypes'][TAXA_LOOKUP[p2]]['calls'][:N_SNPS].astype(np.float32)
    snp1[snp1 == -1] = 0
    snp2[snp2 == -1] = 0
  env = loc_data[FEAT_COLS].mean().values
  snp_c = np.concatenate([snp1, snp2]).reshape(1,-1)
  snp_sc = scaler_snp.transform(snp_c)
  snp_pca = pca.transform(snp_sc)
  x = np.concatenate([snp_pca, env.reshape(1,-1)], axis=1)

  x_sc = scaler_final.transform(x)
  pred = round(rf.predict(x_sc)[0], 2)
  if verbose:
    cat = ('High' if pred >= 170 else
           'Medium' if pred>= 150 else
           'Low')
    print(f"{parent1} × {parent2} @ {location}")
    print(f"Predicted: {pred} bu/A {cat}")
  return pred
# quick test
predict_yield('B73', 'Mo17', 'ILH1')

B73 × Mo17 @ ILH1
Predicted: 174.03 bu/A High


np.float64(174.03)

In [71]:
# test and screen crosses
print('Test Predictions')
print(f"{'Female':15} {'Male':10} {'Location':12} {'Yield':>10}")

test_list = [
    ('B73',  'Mo17',   'ILH1'),
    ('B73',  'Mo17',   'GAH1'),
    ('A632', '3IIH6',  'WIH1'),
    ('A632', '3IIH6',  'GAH1'),
    ('Oh43', 'Mo17',   'IAH4'),
    ('B97',  '3IIH6',  'MNH1'),
]

for p1, p2, loc in test_list:
  pred = predict_yield(p1, p2, loc, verbose=False)
  if pred:
    cat = ('High' if pred >= 170 else
           'Medium' if pred >= 150 else 'Low')
    print(f"{p1:15} {p2:10} {loc:12} "
              f"{pred:>8.2f}  {cat}")

print()
# finding best location for B73/Mo17
print('Best location for b73 × Mo17:')
locations = sorted(df['Field-Location'].unique())

results = []

for loc in locations:
  pred = predict_yield('B73', 'Mo17', loc, verbose=False)
  if pred:
    results.append({'Location': loc, 'Yield': pred})

df_locs = pd.DataFrame(results).sort_values('Yield', ascending=False).reset_index(drop=True)

print(f"{'Rank':>4}{'Location':15}{'Yield':>12}")
for i, row in df_locs.head(5).iterrows():
  print(f"{i+1:>4} {row['Location']:15} "
          f"{row['Yield']:>10.2f} bu/A")

Test Predictions
Female          Male       Location          Yield
B73             Mo17       ILH1           174.03  High
B73             Mo17       GAH1           108.83  Low
A632            3IIH6      WIH1           178.64  High
A632            3IIH6      GAH1           109.23  Low
Oh43            Mo17       IAH4           196.05  High
B97             3IIH6      MNH1           151.04  Medium

Best location for b73 × Mo17:
RankLocation              Yield
   1 WIH2                218.88 bu/A
   2 WIH1                211.77 bu/A
   3 NYH3                203.59 bu/A
   4 ONH1                201.79 bu/A
   5 DEH1                201.62 bu/A


In [72]:
# run this cell after complete run. next time run this instead of all cells
def quick_load():
  global rf, pca, scaler_snp, scaler_final
  global X_snp, X_env, X_final, y, df

  X_snp = np.load(BASE+'X_snp.npy')
  X_env = np.load(BASE+'X_env.npy')
  X_final = np.load(BASE+'X_final.npy')
  y = np.load(BASE+'y.npy')
  df = pd.read_csv(BASE+'df_raw.csv')
  rf = pickle.load(open(BASE+'best_model.pkl', 'rb'))
  pca = pickle.load(open(BASE+'pca.pkl', 'rb'))
  scaler_snp = pickle.load(open(BASE+'scaler_snp.pkl', 'rb'))
  scaler_final = pickle.load(open(BASE+'scaler_final.pkl', 'rb'))

  print('Everyrhing loaded')
  print(f'Samples: {len(y):,}')
  print(f'Features: {X_final.shape[1]}')
  print(f'CV r2: 0.572')
  print(f'Test R2: 0.635')

  # uncomment after session reset
quick_load()


Everyrhing loaded
Samples: 2,867
Features: 27
CV r2: 0.572
Test R2: 0.635


In [73]:
# ── Fix and load all files ─────────────────────────────
import numpy as np
import pandas as pd
import pickle

BASE = '/content/drive/My Drive/Dl Models/geno_pheno/'

# Load arrays
X_snp   = np.load(BASE + 'X_snp.npy')
X_env   = np.load(BASE + 'X_env.npy')
X_final = np.load(BASE + 'X_final.npy')
y       = np.load(BASE + 'y.npy')
df      = pd.read_csv(BASE + 'df_raw.csv')

# Load model files
rf           = pickle.load(open(BASE + 'best_model.pkl', 'rb'))
pca          = pickle.load(open(BASE + 'pca.pkl',         'rb'))
scaler_snp   = pickle.load(open(BASE + 'scaler_snp.pkl',  'rb'))
scaler_final = pickle.load(open(BASE + 'scaler_final.pkl','rb'))

print("Loaded successfully")
print(f"   X_snp:   {X_snp.shape}")
print(f"   X_env:   {X_env.shape}")
print(f"   X_final: {X_final.shape}")
print(f"   y:       {y.shape}  value: {y[:3]}")
print(f"   df:      {df.shape}")

Loaded successfully
   X_snp:   (2867, 10000)
   X_env:   (2867, 17)
   X_final: (2867, 27)
   y:       (2867,)  value: [20.5  52.67 63.3 ]
   df:      (11746, 24)


## Deploying model on streamlit

In [74]:
# Define predict_yield
def predict_yield(parent1, parent2, location, verbose=False):
    p1 = match_id(parent1)
    p2 = match_id(parent2)
    if not p1 or not p2: return None
    if p1 not in TAXA_LOOKUP or p2 not in TAXA_LOOKUP: return None
    loc_data = df[df['Field-Location'] == location]
    if len(loc_data) == 0: return None
    with h5py.File(GEN, 'r') as f:
        snp1 = f['Genotypes'][TAXA_LOOKUP[p1]]['calls'][:N_SNPS].astype(np.float32)
        snp2 = f['Genotypes'][TAXA_LOOKUP[p2]]['calls'][:N_SNPS].astype(np.float32)
        snp1[snp1==-1]=0; snp2[snp2==-1]=0
    env = loc_data[FEAT_COLS].mean().values
    snp_c = np.concatenate([snp1,snp2]).reshape(1,-1)
    snp_pca = pca.transform(scaler_snp.transform(snp_c))
    x_sc = scaler_final.transform(
                  np.concatenate([snp_pca, env.reshape(1,-1)], axis=1)
              )
    return round(float(rf.predict(x_sc)[0]), 2)

print("predict_yield defined")
print("Test:", predict_yield('B73','Mo17','ILH1'))

predict_yield defined
Test: 174.03


In [ ]:
# Run this in Colab
# For Pre-compute predictions for all parent combinations

print("Computing all predictions...")

all_p1    = sorted(df['clean_p1'].dropna().unique())
all_p2    = sorted(df['clean_p2'].dropna().unique())
locations = sorted(df['Field-Location'].unique())

results = []
total   = len(all_p1) * len(all_p2) * len(locations)
count   = 0

for p1 in all_p1:
    for p2 in all_p2:
        for loc in locations:
            pred = predict_yield(
                p1, p2, loc, verbose=False
            )
            if pred:
                results.append({
                    'Female': p1,
                    'Male': p2,
                    'Location': loc,
                    'Yield': pred
                })
            count += 1
            if count % 10000 == 0:
                print(f"{count:,}/{total:,} done...")

df_preds = pd.DataFrame(results)
df_preds.to_csv(BASE + 'all_predictions.csv', index=False)

print(f"\nDone!")
print(f"Total predictions: {len(df_preds):,}")
print(f"File size: "
      f"{os.path.getsize(BASE+'all_predictions.csv')/1024/1024:.1f} MB")
print(df_preds.head())

In [77]:
# checking if prediction.csv file is created in drive
print(os.path.exists(BASE + 'all_predictions.csv'))
print(os.path.getsize(BASE + 'all_predictions.csv')/1024/1024, "MB")

True
2.5463714599609375 MB


In [78]:

# CREATEING app.py FOR STREAMLIT CLOUD

app_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(
    page_title="Maize Yield Predictor",
    page_icon="🌽",
    layout="wide"
)

@st.cache_data
def load_data():
    return pd.read_csv("all_predictions.csv")

df = load_data()

# Get unique values
females   = sorted(df["Female"].unique().tolist())
males     = sorted(df["Male"].unique().tolist())
locations = sorted(df["Location"].unique().tolist())

def lookup(p1, p2, loc):
    res = df[
        (df["Female"]   == p1) &
        (df["Male"]     == p2) &
        (df["Location"] == loc)
    ]
    if len(res) == 0:
        res = df[
            (df["Female"]   == p2) &
            (df["Male"]     == p1) &
            (df["Location"] == loc)
        ]
    return round(float(res.iloc[0]["Yield"]), 2) \
           if len(res) > 0 else None

def category(y):
    if y >= 170: return "🟢 High"
    if y >= 150: return "🟡 Medium"
    return "🔴 Low"

# Header
st.title("🌽 Maize Yield Predictor")
st.markdown(
    "**AI-powered yield prediction using "
    "genomics + environment | G2F 2017**"
)
st.markdown("---")

# Metrics
col1, col2, col3, col4 = st.columns(4)
col1.metric("CV R²",     "0.572")
col2.metric("Test R²",   "0.635")
col3.metric("Samples",   "2,867")
col4.metric("Locations", "23")
st.markdown("---")

# Sidebar inputs
st.sidebar.title("🔬 Parameters")
default_f = females.index("B73")  if "B73"  in females else 0
default_m = males.index("Mo17")   if "Mo17" in males   else 0

female   = st.sidebar.selectbox("Female Parent",
                                  females, index=default_f)
male     = st.sidebar.selectbox("Male Parent",
                                  males,   index=default_m)
location = st.sidebar.selectbox("Location", locations)

# Tabs
tab1, tab2, tab3, tab4 = st.tabs([
    "🔮 Predict",
    "📍 Best Location",
    "🏆 Best Cross",
    "🔄 G×E Analysis"
])

# TAB 1: Single Prediction
with tab1:
    st.subheader(
        f"Prediction: {female} × {male} @ {location}"
    )
    pred = lookup(female, male, location)

    if pred:
        col1, col2, col3 = st.columns(3)
        col1.metric("Predicted Yield", f"{pred} bu/A")
        col2.metric("Female Parent",   female)
        col3.metric("Male Parent",     male)

        cat = category(pred)
        if "High"   in cat: st.success(f"{cat} Yield")
        elif "Medium" in cat: st.warning(f"{cat} Yield")
        else:                 st.error(f"{cat} Yield")

        # Gauge
        fig = go.Figure(go.Indicator(
            mode="gauge+number+delta",
            value=pred,
            title={"text": "Predicted Yield (bu/A)"},
            delta={"reference": df["Yield"].mean()},
            gauge={
                "axis": {
                    "range": [
                        df["Yield"].min(),
                        df["Yield"].max()
                    ]
                },
                "bar": {"color": "steelblue"},
                "steps": [
                    {"range": [df["Yield"].min(), 150],
                     "color": "#ffcccc"},
                    {"range": [150, 170],
                     "color": "#ffffcc"},
                    {"range": [170, df["Yield"].max()],
                     "color": "#ccffcc"}
                ]
            }
        ))
        fig.update_layout(height=350)
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.warning("Combination not found in database")

# TAB 2: Best Location
with tab2:
    st.subheader(f"Best Locations for {female} × {male}")
    rows = []
    for loc in locations:
        p = lookup(female, male, loc)
        if p:
            rows.append({
                "Location": loc,
                "Yield":    p,
                "Category": category(p)
            })

    if rows:
        res_df = pd.DataFrame(rows)\
                   .sort_values("Yield", ascending=False)\
                   .reset_index(drop=True)
        res_df.index += 1

        best = res_df.iloc[0]
        st.success(
            f"🏆 Best: **{best['Location']}** "
            f"→ {best['Yield']} bu/A"
        )

        fig = px.bar(
            res_df,
            x="Location", y="Yield",
            color="Yield",
            color_continuous_scale="RdYlGn",
            title=f"{female} × {male} — Yield by Location",
            text="Yield",
            template="plotly_white"
        )
        fig.update_traces(
            texttemplate="%{text:.1f}",
            textposition="outside"
        )
        fig.update_layout(
            xaxis_tickangle=-45,
            height=450,
            showlegend=False
        )
        st.plotly_chart(fig, use_container_width=True)
        st.dataframe(res_df, use_container_width=True)

        csv = res_df.to_csv(index=False)
        st.download_button(
            "📥 Download CSV", csv,
            f"best_locations_{female}_{male}.csv"
        )

# TAB 3: Best Cross at Location
with tab3:
    st.subheader(f"Top Crosses at {location}")
    top_n = st.slider("Show top N", 5, 50, 20)

    cross_df = df[df["Location"] == location].copy()
    cross_df["Cross"] = (cross_df["Female"] +
                          " × " + cross_df["Male"])
    cross_df = cross_df\
               .sort_values("Yield", ascending=False)\
               .head(top_n)\
               .reset_index(drop=True)
    cross_df.index += 1

    fig = px.bar(
        cross_df,
        x="Yield", y="Cross",
        orientation="h",
        color="Yield",
        color_continuous_scale="RdYlGn",
        title=f"Top {top_n} Crosses at {location}",
        text="Yield",
        template="plotly_white"
    )
    fig.update_traces(
        texttemplate="%{text:.1f}",
        textposition="outside"
    )
    fig.update_layout(
        height=max(400, top_n*25),
        showlegend=False,
        yaxis={"categoryorder": "total ascending"}
    )
    st.plotly_chart(fig, use_container_width=True)

    csv = cross_df.to_csv(index=False)
    st.download_button(
        "📥 Download CSV", csv,
        f"top_crosses_{location}.csv"
    )

# TAB 4: G×E Analysis
with tab4:
    st.subheader("G×E Interaction Analysis")

    sel_females = st.multiselect(
        "Select Female Parents",
        females,
        default=females[:3]
    )
    fixed_male = st.selectbox(
        "Fixed Male Parent", males,
        index=males.index("Mo17") if "Mo17" in males else 0
    )

    if sel_females:
        ge_df = df[
            (df["Female"].isin(sel_females)) &
            (df["Male"] == fixed_male)
        ].copy()
        ge_df["Hybrid"] = (ge_df["Female"] +
                            " × " + ge_df["Male"])

        if len(ge_df) > 0:
            fig = px.line(
                ge_df,
                x="Location", y="Yield",
                color="Hybrid",
                markers=True,
                title="G×E Interaction",
                template="plotly_white"
            )
            fig.add_hline(
                y=ge_df["Yield"].mean(),
                line_dash="dash",
                annotation_text="Average"
            )
            fig.update_layout(
                height=500,
                xaxis_tickangle=-45
            )
            st.plotly_chart(fig, use_container_width=True)

            pivot = ge_df.pivot(
                index="Hybrid",
                columns="Location",
                values="Yield"
            )
            fig2 = px.imshow(
                pivot,
                color_continuous_scale="RdYlGn",
                title="Yield Heatmap",
                text_auto=".1f",
                template="plotly_white"
            )
            fig2.update_layout(height=400)
            st.plotly_chart(fig2, use_container_width=True)

# Footer
st.markdown("---")
st.markdown(
    "Built with G2F 2017 | Random Forest | "
    "CV R² = 0.572 | **Abdul Manan**"
)
'''

with open('/content/app.py', 'w') as f:
    f.write(app_code)

# Also save to Drive
with open(BASE + 'app.py', 'w') as f:
    f.write(app_code)

print("app.py created")

app.py created
